In [ ]:
!pip install catboost paddleocr paddlepaddle sentencepiece protobuf pyarrow


In [ ]:
import os
import re
import gc
import json
import math
import random
import numpy as np
import pandas as pd
from PIL import Image

# 1. Защита от сбоев в библиотеках HuggingFace и PaddleOCR
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"] = "0"
import torch
import cv2
import paddle
from catboost import CatBoostClassifier
from paddleocr import PaddleOCR
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoImageProcessor,
    AutoModelForImageClassification
)

# 3. Настройка вычислительного устройства (GPU T4 / CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Используемое устройство:", DEVICE)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

try:
    if torch.cuda.is_available():
        paddle.device.set_device("gpu")
    else:
        paddle.device.set_device("cpu")
except Exception:
    pass

print("\n✓ Все системные библиотеки успешно загружены!\n")


In [ ]:
# --- 3.1. Анализатор длины и дисперсии предложений ---
def extract_sentence_length_features(text):
    if not text or len(text.strip()) < 10:
        return 0.0, 0.0, 0.0

    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    if not sentences:
        return 0.0, 0.0, 0.0

    sentence_lengths = [len(s.split()) for s in sentences]
    avg_len = float(np.mean(sentence_lengths))
    len_var = float(np.var(sentence_lengths)) if len(sentence_lengths) > 1 else 0.0
    short_sentences = sum(1 for l in sentence_lengths if l < 4)
    short_ratio = float(short_sentences / len(sentences))

    return avg_len, len_var, short_ratio


# --- 3.2. Эвристический модуль No-AI-Slop (поиск штампов ИИ) ---
SLOP_BUZZWORDS_EN = {"delve", "tapestry", "crucial", "pivotal", "testament", "beacon", "multifaceted", "paramount", "harness", "foster"}
SLOP_BUZZWORDS_RU = {"погрузиться", "гобелен", "неотъемлемый", "краеугольный", "симфония", "свидетельство", "многогранный", "бесшовный"}

THROAT_CLEARING_PATTERNS = [
    r"\bhere'?s\s+the\s+thing\b", r"\blet'?s\s+dive\s+in\b", r"\bin\s+today'?s\s+fast-paced\b",
    r"\bit'?s\s+important\s+to\s+note\b", r"\bважно\s+(?:понимать|отметить)\b", r"\bстоит\s+отметить\b"
]

BINARY_CONTRAST_PATTERNS = [
    r"\b(?:it'?s|this\s+is)\s+not\s+(?:just|only)\b.*?\b(?:it'?s|it\s+is)\b",
    r"\bnot\s+only\b.*?\bbut\s+also\b", r"\bне\s+просто\b.*?\bа\b", r"\bне\s+только\b.*?\bно\s+и\b"
]

def extract_ai_slop_features(text):
    if not text or len(text.strip()) < 10:
        return {"slop_buzzwords": 0, "slop_throat_clearing": 0, "slop_binary_contrast": 0, "slop_em_dashes": 0.0}

    text_lower = text.lower()
    words = re.findall(r'\b\w+\b', text_lower)
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    num_sentences = max(1, len(sentences))

    buzzwords_hit = sum(1 for w in words if w in SLOP_BUZZWORDS_EN or w in SLOP_BUZZWORDS_RU)
    first_two_sent = " ".join(sentences[:2]).lower() if sentences else text_lower
    throat_hit = int(any(re.search(p, first_two_sent) for p in THROAT_CLEARING_PATTERNS))
    contrast_hit = sum(len(re.findall(p, text_lower)) for p in BINARY_CONTRAST_PATTERNS)
    em_dashes_ratio = float(len(re.findall(r'[—–]|--', text)) / num_sentences)

    return {
        "slop_buzzwords": buzzwords_hit,
        "slop_throat_clearing": throat_hit,
        "slop_binary_contrast": contrast_hit,
        "slop_em_dashes": round(em_dashes_ratio, 4)
    }

print("✓ Лингвистические экстракторы скомпилированы!")


In [ ]:
class BinocularsFeatureExtractor:
    def __init__(self, observer_name="Qwen/Qwen2.5-1.5B", performer_name="Qwen/Qwen2.5-1.5B-Instruct", device=DEVICE):
        self.device = torch.device(device)
        print(f"Загрузка моделей Binoculars: {observer_name} & {performer_name}...")

        self.tokenizer = AutoTokenizer.from_pretrained(observer_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        dtype = torch.float16 if self.device.type == "cuda" else torch.float32

        self.observer = AutoModelForCausalLM.from_pretrained(
            observer_name, torch_dtype=dtype, low_cpu_mem_usage=True
        ).to(self.device)

        self.performer = AutoModelForCausalLM.from_pretrained(
            performer_name, torch_dtype=dtype, low_cpu_mem_usage=True
        ).to(self.device)

        self.observer.eval()
        self.performer.eval()
        self.observer.config.use_cache = False
        self.performer.config.use_cache = False

        gc.collect()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()
        print("✓ Текстовые модели Qwen 2.5 успешно загружены!\n")

    @torch.inference_mode()
    def calculate_score_from_ids(self, input_ids):
        attention_mask = torch.ones_like(input_ids)

        obs_output = self.observer(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        obs_logits = obs_output.logits[..., :-1, :].float()
        shift_labels = input_ids[..., 1:]

        obs_log_probs = torch.log_softmax(obs_logits, dim=-1)
        obs_gathered = torch.gather(obs_log_probs, dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)
        log_ppl = -obs_gathered.mean().item()
        del obs_output

        perf_output = self.performer(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        perf_logits = perf_output.logits[..., :-1, :].float()

        perf_log_probs = torch.log_softmax(perf_logits, dim=-1)
        perf_probs = torch.exp(perf_log_probs)
        cross_entropy = -(perf_probs * obs_log_probs).sum(dim=-1)
        log_x_ppl = cross_entropy.mean().item()

        del obs_logits, perf_logits, obs_log_probs, perf_log_probs, perf_probs, cross_entropy, perf_output, attention_mask

        return float(log_ppl / log_x_ppl if log_x_ppl != 0 else 1.0)

    @torch.inference_mode()
    def extract_window_features(self, text, window_size=50, step=15):
        """Возвращает строго 4 значения: min, mean, var, most_ai_chunk"""
        if not text or len(text.strip()) < 10:
            return 1.0, 1.0, 0.0, ""

        try:
            inputs = self.tokenizer(text, return_tensors="pt", truncation=False)
            input_ids = inputs["input_ids"].to(self.device)
            total_tokens = input_ids.shape[1]

            if total_tokens <= window_size:
                score = self.calculate_score_from_ids(input_ids)
                decoded_text = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
                del input_ids
                return float(score), float(score), 0.0, decoded_text.strip()

            scores = []
            windows_tokens = []
            for start in range(0, total_tokens - window_size + 1, step):
                w_ids = input_ids[:, start:start + window_size]
                score = self.calculate_score_from_ids(w_ids)
                scores.append(score)
                windows_tokens.append(w_ids)

            del input_ids
            if not scores:
                return 1.0, 1.0, 0.0, ""

            min_idx = int(np.argmin(scores))
            most_ai_chunk = self.tokenizer.decode(windows_tokens[min_idx][0], skip_special_tokens=True).strip()

            scores = np.asarray(scores, dtype=np.float32)
            return float(np.min(scores)), float(np.mean(scores)), float(np.var(scores)), most_ai_chunk

        except Exception as e:
            print(f"Ошибка обработки текста: {e}")
            return 1.0, 1.0, 0.0, ""

# Инициализируем текстовый экстрактор
extractor = BinocularsFeatureExtractor(device=DEVICE)


In [ ]:
class VisualOCREngine:
    def __init__(self, lang="ru"):
        print(f"Инициализация движка PaddleOCR (язык: {lang})...")
        # В PaddleOCR 3.x передаем только lang (без устаревших параметров cls и show_log)
        self.ocr = PaddleOCR(lang=lang)
        print("✓ Модуль PaddleOCR успешно загружен!\n")

    def extract_text(self, image_path):
        """Оцифровка изображения с добавлением 30px белого поля для избежания обрезки краевых букв."""
        if not image_path or not os.path.exists(image_path):
            return ""
        try:
            img = cv2.imread(image_path)
            if img is None:
                return ""

            # Добавляем защитные поля со всех сторон (Padding)
            padded_img = cv2.copyMakeBorder(
                img, 30, 30, 30, 30,
                cv2.BORDER_CONSTANT,
                value=[255, 255, 255]
            )

            # Вызов метода predict или ocr
            if hasattr(self.ocr, "predict"):
                result = self.ocr.predict(padded_img)
                if result and len(result) > 0:
                    first_res = result[0]
                    if isinstance(first_res, dict) and "rec_texts" in first_res:
                        return "\n".join(first_res["rec_texts"]).strip()
                    elif hasattr(first_res, "get"):
                        return "\n".join(first_res.get("rec_texts", [])).strip()

            result = self.ocr.ocr(padded_img)
            if not result or result[0] is None:
                return ""

            recognized_lines = []
            for line in result[0]:
                if isinstance(line, (list, tuple)) and len(line) >= 2:
                    text_info = line[1]
                    if isinstance(text_info, (list, tuple)) and len(text_info) >= 1:
                        recognized_lines.append(str(text_info[0]))
                    elif isinstance(text_info, str):
                        recognized_lines.append(str(text_info))

            return "\n".join(recognized_lines).strip()

        except Exception as e:
            print(f"[Предупреждение OCR]: {e}")
            return ""

    def get_multimodal_features(self, image_path, text_extractor):
        raw_image_text = self.extract_text(image_path)
        if len(raw_image_text) >= 10:
            print(f" -> PaddleOCR оцифровал текст:\n\"{raw_image_text[:80]}...\"")
            b_min, b_mean, b_var, _ = text_extractor.extract_window_features(raw_image_text)
            return {
                "has_image_text": 1,
                "ocr_bino_min": b_min,
                "ocr_bino_mean": b_mean,
                "ocr_bino_var": b_var,
                "extracted_image_text": raw_image_text
            }
        else:
            return {
                "has_image_text": 0,
                "ocr_bino_min": 1.0,
                "ocr_bino_mean": 1.0,
                "ocr_bino_var": 0.0,
                "extracted_image_text": ""
            }

ocr_engine = VisualOCREngine(lang="ru")


In [ ]:
class RealSafeImageDetector:
    def __init__(self, model_name="dima806/deepfake_vs_real_image_detection", device=DEVICE):
        self.device = device
        print(f"Загрузка детектора ИИ-графики SAFE ({model_name})...")
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModelForImageClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

        self.fake_label_idx = 0
        for idx, label in self.model.config.id2label.items():
            if any(term in label.lower() for term in ['fake', 'ai', 'synthetic']):
                self.fake_label_idx = int(idx)
                break
        print("✓ Детектор SAFE успешно загружен на GPU!\n")

    def get_safe_score(self, image_path, num_patches=4, patch_size=224):
        """Методология SAFE: нарезка патчей высокого разрешения вместо даунсэмплинга."""
        if not image_path or not os.path.exists(image_path):
            return 0.0
        try:
            image = Image.open(image_path).convert("RGB")
            w, h = image.size
            patches = []
            if w <= patch_size or h <= patch_size:
                patches.append(image.resize((patch_size, patch_size)))
            else:
                patches.append(image.crop(((w - patch_size)//2, (h - patch_size)//2, (w + patch_size)//2, (h + patch_size)//2)))
                for _ in range(num_patches - 1):
                    rx = np.random.randint(0, w - patch_size)
                    ry = np.random.randint(0, h - patch_size)
                    patches.append(image.crop((rx, ry, rx + patch_size, ry + patch_size)))

            scores = []
            with torch.no_grad():
                for patch in patches:
                    inputs = self.processor(images=patch, return_tensors="pt").to(self.device)
                    outputs = self.model(**inputs)
                    probs = torch.softmax(outputs.logits, dim=-1)[0]
                    scores.append(probs[self.fake_label_idx].item())

            return round(float(np.mean(scores)), 4)
        except Exception as e:
            print(f"[Ошибка SAFE]: {e}")
            return 0.0

safe_detector = RealSafeImageDetector(device=DEVICE)


In [ ]:
def build_final_parquet(jsonl_input_path, parquet_output_path):
    if not os.path.exists(jsonl_input_path):
        print(f"[Ошибка] Входной файл {jsonl_input_path} не найден на левой панели!")
        return None

    records = []
    with open(jsonl_input_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    print(f"Загружено записей из JSONL: {len(records)}")
    ml_data = []
    processed_human_emails = set()

    for idx, row in enumerate(records):
        email_id = str(row.get('email_id'))
        var_idx = row.get('variant_index', 0)
        orig_body = str(row.get('original_body', ''))
        ai_body = str(row.get('ai_mutated_body', ''))

        # Класс 0: Человек (считаем строго 1 раз на email_id без дубликатов)
        if email_id not in processed_human_emails:
            o_min, o_mean, o_var, _ = extractor.extract_window_features(orig_body)
            o_avg_len, o_len_var, o_short_ratio = extract_sentence_length_features(orig_body)
            o_slop = extract_ai_slop_features(orig_body)

            ml_data.append({
                'email_id': email_id, 'variant_index': -1, 'text_type': 'Human', 'is_ai': 0,
                'bino_min': o_min, 'bino_mean': o_mean, 'bino_var': o_var,
                'avg_sent_len': o_avg_len, 'sent_len_var': o_len_var, 'short_sent_ratio': o_short_ratio,
                'slop_buzzwords': o_slop['slop_buzzwords'],
                'slop_throat_clearing': o_slop['slop_throat_clearing'],
                'slop_binary_contrast': o_slop['slop_binary_contrast'],
                'slop_em_dashes': o_slop['slop_em_dashes']
            })
            processed_human_emails.add(email_id)

        # Класс 1: ИИ (считаем для каждого варианта мутации)
        a_min, a_mean, a_var, _ = extractor.extract_window_features(ai_body)
        a_avg_len, a_len_var, a_short_ratio = extract_sentence_length_features(ai_body)
        a_slop = extract_ai_slop_features(ai_body)

        ml_data.append({
            'email_id': email_id, 'variant_index': var_idx, 'text_type': 'AI', 'is_ai': 1,
            'bino_min': a_min, 'bino_mean': a_mean, 'bino_var': a_var,
            'avg_sent_len': a_avg_len, 'sent_len_var': a_len_var, 'short_sent_ratio': a_short_ratio,
            'slop_buzzwords': a_slop['slop_buzzwords'],
            'slop_throat_clearing': a_slop['slop_throat_clearing'],
            'slop_binary_contrast': a_slop['slop_binary_contrast'],
            'slop_em_dashes': a_slop['slop_em_dashes']
        })

        if idx % 10 == 0 and DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    df = pd.DataFrame(ml_data)
    df.to_parquet(parquet_output_path, engine='pyarrow', index=False)
    print(f"[Успешно] Матрица признаков сохранена в: {parquet_output_path}")
    return df

DATASET_PATH = "mutated_spam_dataset.jsonl"
OUTPUT_PARQUET = "final_features.parquet"

df_features = build_final_parquet(DATASET_PATH, OUTPUT_PARQUET)

In [ ]:
feature_cols = [
    'bino_min', 'bino_mean', 'bino_var',
    'avg_sent_len', 'sent_len_var', 'short_sent_ratio',
    'slop_buzzwords', 'slop_throat_clearing', 'slop_binary_contrast', 'slop_em_dashes'
]

def train_catboost_detector(df):
    if df is None or len(df) == 0:
        print("[Ошибка] Нет данных для обучения.")
        return None
    df_clean = df.dropna(subset=feature_cols + ['is_ai'])
    X = df_clean[feature_cols]
    y = df_clean['is_ai']

    model = CatBoostClassifier(
        iterations=200,
        depth=4,
        learning_rate=0.03,
        auto_class_weights='Balanced',
        verbose=False,
        random_seed=42
    )
    model.fit(X, y)
    print("✓ Модель CatBoost успешно обучена на матрице признаков!\n")
    return model

cb_model = train_catboost_detector(df_features)


In [ ]:
def extract_ai_generated_segments(text, window_size=50, step=15, confidence_threshold=75.0):
    if not text or len(text.strip()) < 10:
        return []

    try:
        inputs = extractor.tokenizer(text, return_tensors="pt").to(extractor.device)
        input_ids = inputs["input_ids"][0]
        total_tokens = len(input_ids)

        B_0 = 0.9015  # Порог Binoculars для Qwen
        k = 35.0      # Коэффициент крутизны

        detected_windows = []

        def calc_confidence(score_val):
            return 1.0 / (1.0 + math.exp(-k * (B_0 - score_val))) * 100.0

        if total_tokens <= window_size:
            score = extractor.calculate_score_from_ids(input_ids.unsqueeze(0))
            conf = calc_confidence(score)
            if conf >= confidence_threshold:
                decoded = extractor.tokenizer.decode(input_ids, skip_special_tokens=True)
                detected_windows.append({"start": 0, "end": total_tokens, "text": decoded.strip(), "confidence": conf})
        else:
            for i in range(0, total_tokens - window_size + 1, step):
                w_ids = input_ids[i: i + window_size]
                score = extractor.calculate_score_from_ids(w_ids.unsqueeze(0))
                conf = calc_confidence(score)

                if conf >= confidence_threshold:
                    decoded = extractor.tokenizer.decode(w_ids, skip_special_tokens=True)
                    detected_windows.append({"start": i, "end": i + window_size, "text": decoded.strip(), "confidence": conf})

        if not detected_windows:
            return []

        # Склеивание перекрывающихся зон
        detected_windows.sort(key=lambda x: x["start"])
        merged = [detected_windows[0]]

        for current in detected_windows[1:]:
            prev = merged[-1]
            if current["start"] <= prev["end"]:
                prev["end"] = max(prev["end"], current["end"])
                prev["confidence"] = max(prev["confidence"], current["confidence"])
                combined_tokens = input_ids[prev["start"]: prev["end"]]
                prev["text"] = extractor.tokenizer.decode(combined_tokens, skip_special_tokens=True).strip()
            else:
                merged.append(current)

        return merged

    except Exception as e:
        print(f"Ошибка локализации: {e}")
        return []


In [ ]:
def analyze_email(raw_text, image_path=None):
    # 1. Текстовые расчеты
    b_min, b_mean, b_var, _ = extractor.extract_window_features(raw_text)
    avg_len, len_var, short_ratio = extract_sentence_length_features(raw_text)
    slop = extract_ai_slop_features(raw_text)

    X_infer = pd.DataFrame([{
        'bino_min': b_min, 'bino_mean': b_mean, 'bino_var': b_var,
        'avg_sent_len': avg_len, 'sent_len_var': len_var, 'short_sent_ratio': short_ratio,
        'slop_buzzwords': slop['slop_buzzwords'],
        'slop_throat_clearing': slop['slop_throat_clearing'],
        'slop_binary_contrast': slop['slop_binary_contrast'],
        'slop_em_dashes': slop['slop_em_dashes']
    }])

    prob_ai = cb_model.predict_proba(X_infer)[0][1] if cb_model else 0.0
    is_ai = int(cb_model.predict(X_infer)[0]) if cb_model else 0
    suspicious_segments = extract_ai_generated_segments(raw_text, confidence_threshold=70.0)

    # 2. Визуальный анализ вложения (PaddleOCR + SAFE)
    visual_report = None
    if image_path and os.path.exists(image_path):
        ocr_res = ocr_engine.get_multimodal_features(image_path, extractor)
        real_img_ai_score = safe_detector.get_safe_score(image_path)

        visual_report = {
            "has_image_text": ocr_res["has_image_text"],
            "extracted_text": ocr_res["extracted_image_text"],
            "ocr_bino_min": ocr_res["ocr_bino_min"],
            "img_safe_score": real_img_ai_score,
            "is_image_ai_generated": bool(real_img_ai_score >= 0.5)
        }

    return {
        "verdict": "AI Generated" if is_ai == 1 else "Human Written",
        "ai_probability": f"{prob_ai * 100:.2f}%",
        "features": {
            "bino_min": round(b_min, 4),
            "bino_var": round(b_var, 6),
            "avg_sent_len": round(avg_len, 2)
        },
        "visual_attachment": visual_report,
        "suspicious_segments_count": len(suspicious_segments),
        "segments": suspicious_segments
    }

# --- ТЕСТ-ДРАЙВ СИСТЕМЫ ---
test_image = "tttt.png" if os.path.exists("tttt.png") else None
sample_text = "Dear user, we detected an unauthorized login to your account. Please check the attached screenshot."

print("=== ТЕСТ МУЛЬТИМОДАЛЬНОГО ИНФЕРЕНСА ===")
result = analyze_email(sample_text, image_path=test_image)
print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
# КОНВЕЙЕР СБОРКИ МАСТЕР-ДАТАСЕТА (4 КВАДРАНТА + МУЛЬТИСИГНАЛЫ)
# =====================================================================
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

# Проверяем доступность устройства и экстрактора
if 'DEVICE' not in globals():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Используемое устройство:", DEVICE)

# %%
# 1. ЗАГРУЗКА И СБАЛАНСИРОВАННАЯ ФИЛЬТРАЦИЯ ИЗ MeAJOR
print("Загрузка исторической базы MeAJOR...")
raw_dataset = load_dataset("simlab-vs/meajor_cleaned_preprocessed")
df_meajor = raw_dataset["train"].to_pandas()

print(f"Всего в MeAJOR: {len(df_meajor)} писем.")

# Выбираем размер выборки для комфортной работы в Colab (например, 200 исходных писем)
SAMPLE_SIZE_PER_CLASS = 100

# Извлекаем легитимную почту (Ham) и реальный человеческий спам (Spam)
human_ham = df_meajor[df_meajor['label'] == 0].sample(n=SAMPLE_SIZE_PER_CLASS, random_state=42)
human_spam = df_meajor[df_meajor['label'] == 1].sample(n=SAMPLE_SIZE_PER_CLASS, random_state=42)

print(f"Отобрано человека: {len(human_ham)} Ham (деловые) и {len(human_spam)} Spam (фишинг).")

# %%
# 2. ФУНКЦИЯ ЭКСТРАКЦИИ ВСЕХ ПРИЗНАКОВ ПИСЬМА (ТЕКСТ + HTML)
def extract_email_features(email_id, variant_idx, text_type, is_ai, plain_text, raw_html=""):
    """
    Рассчитывает полный вектор признаков: Binoculars + Синтаксис + Slop + HTML
    """
    # А. Лингвистический Binoculars (Qwen 2.5)
    b_min, b_mean, b_var, most_ai_seg = extractor.extract_window_features(plain_text)

    # Б. Синтаксис предложений
    avg_len, len_var, short_ratio = extract_sentence_length_features(plain_text)

    # В. Эвристики No-AI-Slop
    slop = extract_ai_slop_features(plain_text)

    # Г. Анализ HTML (если разметка передана)
    tag_count = len(re.findall(r'<[^>]+>', raw_html)) if raw_html else 0
    link_count = len(re.findall(r'href\s*=\s*["\'][^"\']+["\']', raw_html, re.IGNORECASE)) if raw_html else 0
    has_hidden = int(bool(re.search(r'(?:display\s*:\s*none|font-size\s*:\s*0|visibility\s*:\s*hidden)', raw_html, re.IGNORECASE))) if raw_html else 0
    text_to_tag = round(len(plain_text.strip()) / max(1, tag_count), 2)

    return {
        "email_id": str(email_id),
        "variant_index": variant_idx,
        "text_type": text_type,
        "is_ai": int(is_ai),  # ЦЕЛЕВАЯ МЕТКА

        # Binoculars
        "bino_min": b_min,
        "bino_mean": b_mean,
        "bino_var": b_var,

        # Синтаксис
        "avg_sent_len": avg_len,
        "sent_len_var": len_var,
        "short_sent_ratio": short_ratio,

        # Slop эвристики
        "slop_buzzwords": slop["slop_buzzwords"],
        "slop_throat_clearing": slop["slop_throat_clearing"],
        "slop_binary_contrast": slop["slop_binary_contrast"],
        "slop_em_dashes": slop["slop_em_dashes"],

        # HTML структура
        "html_tag_count": tag_count,
        "html_link_count": link_count,
        "html_has_hidden_text": has_hidden,
        "html_text_to_tag_ratio": text_to_tag,

        "most_ai_chunk": most_ai_seg
    }

# %%
# 3. ГЕНЕРАТОР ТРЕХ ГРАДАЦИЙ ИИ-МУТАЦИЙ (Микро, Гибрид, Полный рерайт)
def generate_synthetic_mutations(text, subject=""):
    """
    Генерирует мутации трех видов глубины:
    1. Микро-мутация (только CTA/заголовок)
    2. Гибридный рерайт (один абзац)
    3. Полный рерайт текста
    """
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    if not paragraphs:
        paragraphs = [text]

    # Вариант 0: Микро-мутация (добавляем или меняем CTA в конце)
    cta_variants = [
        "Please review the document and verify your credentials immediately to avoid suspension.",
        "In today's fast-paced environment, security is not just an option, it's a necessity.",
        "Kindly access your account portal via the secure verification link provided."
    ]
    micro_mutated = text + "\n\n" + random.choice(cta_variants)

    # Вариант 1: Гибридный рерайт (переписываем первый или второй абзац)
    hybrid_paragraphs = list(paragraphs)
    if len(hybrid_paragraphs) > 0:
        # Эмулируем ИИ-полировку абзаца с добавлением слоп-структур
        hybrid_paragraphs[0] = (
            "It's important to note that our organization requires strict adherence to security protocols. "
            + hybrid_paragraphs[0]
        )
    hybrid_mutated = "\n\n".join(hybrid_paragraphs)

    # Вариант 2: Полный рерайт (глубокая генерация)
    full_mutated = (
        f"Regarding: {subject}\n\n"
        "Delving into the recent system updates, it is crucial to recognize that maintaining compliance "
        "serves as a testament to our commitment to data integrity. "
        "We must ensure our operational workflows remain seamless, robust, and secure.\n\n"
        + "\n\n".join(paragraphs[1:]) if len(paragraphs) > 1 else text
    )

    return [
        ("AI_Micro", micro_mutated),
        ("AI_Hybrid", hybrid_mutated),
        ("AI_Full", full_mutated)
    ]

# %%
# 4. СБОРКА ИТОГОВОЙ МАТРИЦЫ ПРИЗНАКОВ В PARQUET
print("Начало извлечения признаков по 4 квадрантам...")
master_records = []

# Объединяем отобранную базу человека
selected_human_emails = pd.concat([human_ham, human_spam]).reset_index(drop=True)

for idx, row in tqdm(selected_human_emails.iterrows(), total=len(selected_human_emails), desc="Обработка"):
    email_id = str(row.get('id', idx))
    plain_text = str(row.get('body', ''))
    subject = str(row.get('subject', ''))
    raw_html = str(row.get('html_content', '')) if 'html_content' in row else f"<p>{plain_text}</p>"

    if len(plain_text.strip()) < 20:
        continue

    # Квадрант 1 & 2: ЧЕЛОВЕК (Оригинал, is_ai = 0)
    # Определяем, ham это или spam
    source_label = "Human_Ham" if row['label'] == 0 else "Human_Spam"
    human_row = extract_email_features(
        email_id=email_id,
        variant_idx=-1,
        text_type=source_label,
        is_ai=0,
        plain_text=plain_text,
        raw_html=raw_html
    )
    master_records.append(human_row)

    # Квадрант 3 & 4: ИИ-МУТАЦИИ (is_ai = 1 с разной глубиной)
    mutations = generate_synthetic_mutations(plain_text, subject=subject)
    for v_idx, (mut_type, mutated_text) in enumerate(mutations):
        ai_row = extract_email_features(
            email_id=email_id,
            variant_idx=v_idx,
            text_type=mut_type,
            is_ai=1,
            plain_text=mutated_text,
            raw_html=f"<div style='font-family:sans-serif;'><p>{mutated_text}</p></div>"
        )
        master_records.append(ai_row)

    # Очистка кэша GPU
    if idx % 15 == 0 and DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# Сохранение итогового мастер-датасета
master_df = pd.DataFrame(master_records)
OUTPUT_MASTER_PARQUET = "master_features.parquet"
master_df.to_parquet(OUTPUT_MASTER_PARQUET, engine='pyarrow', index=False)

print(f"\n[Успешно] Собрано строк: {len(master_df)}")
print(f"Баланс классов:\n{master_df['is_ai'].value_counts()}")
print(f"\nРазбиение по типам текстов:\n{master_df['text_type'].value_counts()}")
display(master_df.head(4))